# 结构化输出 Structured output

模型可以按照给定的模式提供响应。
这有助于确保输出易于解析，并可用于后续处理。
LangChain 支持多种模式类型和方法来强制输出结构化数据。

## Pydantic

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel
from typing import Optional

# ----------------------
# 统一模型接口
# ----------------------
model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama",  # 想换OpenAI就改成 openai
    temperature=0
)
# ----------------------
# 定义你要的结构
# ----------------------
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: Optional[str]

# ----------------------
# 本地模型支持的结构化输出
# ----------------------
structured_llm = model.with_structured_output(ContactInfo)

# ----------------------
# 测试提取信息
# ----------------------
result = structured_llm.invoke(
    "Extract contact info: John Doe, john@example.com, (555) 123-4567"
)
print(result)

name='John Doe' email='john@example.com' phone='(555) 123-4567'


## TypedDict
Python TypedDict提供了一种比 Pydantic 模型更简单的替代方案，非常适合不需要运行时验证的情况。

In [4]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


## JSON SChema

In [ ]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


## include_raw=True
消息输出及解析后的结构

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
# {
#     "raw": AIMessage(...),
#     "parsed": Movie(title=..., year=..., ...),
#     "parsing_error": None,
# }

{'raw': AIMessage(content='{ "title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8 }\n\n    \t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b-instruct-q4_K_M', 'created_at': '2026-04-08T02:56:56.6952801Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1838166900, 'load_duration': 117044900, 'prompt_eval_count': 36, 'prompt_eval_duration': 53517800, 'eval_count': 51, 'eval_duration': 772494000, 'logprobs': None, 'model_name': 'qwen2.5:3b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019d6b05-7c46-7d22-b8de-8fdfc2e4a273-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 51, 'total_tokens': 87}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

## 嵌套结构
模式嵌套

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)